# Topic 10 — Gradient Boosting

> **Goal of this notebook:** understand the most powerful idea in tabular ML — building an additive model where each new tree fits the **residual errors** of the current ensemble (gradient descent in function space). We'll build it from scratch on a regression problem.

**Contents**
1. Concept & intuition
2. The mathematics (additive model, pseudo-residuals, learning rate)
3. Key hyperparameters
4. The dataset: California Housing
5. Training & staged performance
6. Evaluation & feature importance
7. Gradient Boosting from scratch (regression)
8. Pros, cons & when to use
9. Summary

## 1. Concept & Intuition

Gradient Boosting also builds trees **sequentially**, but instead of re-weighting samples (AdaBoost), each new tree is trained to predict the **errors left over** by the trees so far.

Think of it as iterative correction:
1. Start with a simple guess (e.g. the mean).
2. Look at the **residuals** (actual − predicted).
3. Fit a small tree to those residuals and add a *shrunken* version of it to the model.
4. Repeat — each round nudges predictions closer to the truth.

The "gradient" name comes from the fact that, for squared-error loss, the residual *is* the negative gradient of the loss — so we're doing **gradient descent in function space**.

## 2. The Mathematics

The model is an **additive ensemble** of $M$ trees built greedily. With learning rate $\nu$:

$$F_0(x) = \bar{y}, \qquad F_m(x) = F_{m-1}(x) + \nu\, h_m(x)$$

At each step we compute the **pseudo-residuals** — the negative gradient of the loss $L$ w.r.t. the current predictions:

$$r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F = F_{m-1}}$$

For squared-error loss $L = \tfrac12 (y - F)^2$, this is simply $r_{im} = y_i - F_{m-1}(x_i)$ — the ordinary residual. The new tree $h_m$ is fit to predict these residuals.

The **learning rate** $\nu$ (shrinkage) scales each tree's contribution; smaller $\nu$ needs more trees but generalises better.

## 3. Key Hyperparameters

| Parameter | Effect |
|-----------|--------|
| **n_estimators** | number of boosting rounds (trees). |
| **learning_rate** | shrinkage per tree; lower = more robust, needs more trees. |
| **max_depth** | size of each tree (typically shallow, 2–4). |
| **subsample** | fraction of rows per tree (<1 → stochastic gradient boosting). |

**n_estimators × learning_rate** is the central trade-off; tune them together.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Dataset: California Housing

A regression task — predict median house value — where boosting clearly beats a single tree.

In [ ]:
data = fetch_california_housing()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 5. Training & Staged Performance

`staged_predict` lets us see test error fall as trees are added.

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1,
                               max_depth=3, random_state=42)
gbr.fit(X_train, y_train)

test_rmse = [np.sqrt(mean_squared_error(y_test, p))
             for p in gbr.staged_predict(X_test)]
plt.plot(range(1, len(test_rmse) + 1), test_rmse)
plt.xlabel('number of trees'); plt.ylabel('test RMSE')
plt.title('Gradient Boosting: error drops as trees are added'); plt.show()
print('Final test RMSE:', round(test_rmse[-1], 4), '| R2:', round(gbr.score(X_test, y_test), 4))

## 6. Evaluation & Feature Importance

In [ ]:
imp = gbr.feature_importances_
order = np.argsort(imp)
plt.barh([data.feature_names[i] for i in order], imp[order])
plt.xlabel('importance'); plt.title('Feature importance (Gradient Boosting)'); plt.show()
print('Most important feature:', data.feature_names[int(np.argmax(imp))])

## 7. Gradient Boosting From Scratch (Regression)

The whole algorithm: start at the mean, then repeatedly fit a tree to the residuals and add a shrunken version.

In [ ]:
class GradientBoostingScratch:
    def __init__(self, n_estimators=300, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.lr = learning_rate
        self.max_depth = max_depth
    def fit(self, X, y):
        self.F0 = y.mean()
        Fm = np.full(len(y), self.F0)
        self.trees = []
        for _ in range(self.n_estimators):
            residual = y - Fm                      # negative gradient (squared loss)
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residual)
            Fm += self.lr * tree.predict(X)
            self.trees.append(tree)
        return self
    def predict(self, X):
        pred = np.full(X.shape[0], self.F0)
        for tree in self.trees:
            pred += self.lr * tree.predict(X)
        return pred

scratch = GradientBoostingScratch(n_estimators=300, learning_rate=0.1, max_depth=3)
scratch.fit(X_train, y_train)
sp = scratch.predict(X_test)
print('From-scratch R2:', round(r2_score(y_test, sp), 4),
      '| RMSE:', round(np.sqrt(mean_squared_error(y_test, sp)), 4))
print('scikit-learn R2:', round(gbr.score(X_test, y_test), 4))

## 8. Pros, Cons & When to Use

**Pros**
- State-of-the-art accuracy on **tabular** data.
- Flexible — supports many loss functions (regression, classification, ranking).
- Captures complex non-linear interactions.

**Cons**
- Sequential → slower to train; many hyperparameters to tune.
- Can overfit if `n_estimators` is too high for the `learning_rate` (use early stopping).
- Less interpretable than a single tree.

**When to use**
- Almost any structured/tabular prediction problem where accuracy matters.
- When you can afford tuning — XGBoost/LightGBM (next topic) are optimised versions.

## 9. Summary

- Gradient Boosting builds an **additive** model where each tree fits the **residuals** (negative gradient) of the current ensemble.
- The **learning rate** shrinks each tree's contribution, trading off with the number of trees.
- On California Housing, test error fell smoothly as trees accumulated, and **median income** dominated feature importance.
- Our from-scratch residual-fitting boosting matched scikit-learn, confirming the core idea.